In [39]:
import zipfile
import os

# Your local paths
zip_file_path = r"C:\Users\SM MUKUNTHAN\ipl-powerplay-predictor\data\ipl_male_csv2.zip"
extract_folder = r"C:\Users\SM MUKUNTHAN\ipl-powerplay-predictor\data\ipl2"

# Create the extraction folder if it doesn't exist
os.makedirs(extract_folder, exist_ok=True)

# Extract the files
print("Extracting files...")
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_folder)
print(f"Extraction complete! Files are saved to: {extract_folder}")

Extracting files...
Extraction complete! Files are saved to: C:\Users\SM MUKUNTHAN\ipl-powerplay-predictor\data\ipl2


We train a regression model that uses a baseline of 2 * (runs scored in 3 overs) and use these features:


*   Number of wickets
*   Number of dot-balls


*   Number of boundaries
*   Momentum - (runs in 3rd over) - (run rate in 3 overs)

Then we use data only from 2024 onwards to train the model.
Instead of a normal least squares approxiamation, we use a weighted on that gives more preference to the newer years and hence we arrive at our required beta values




In [ ]:
import os
import pandas as pd
import numpy as np
from scipy.linalg import lstsq

data_path = r"C:\Users\SM MUKUNTHAN\ipl-powerplay-predictor\data\ipl2"

raw_training_data = []

print("Scanning files and extracting features...")
for file in os.listdir(data_path):
    if file.endswith(".csv") and "_info" not in file:
        match_file = os.path.join(data_path, file)

        try:
            df = pd.read_csv(match_file, header=None, engine="python", on_bad_lines="skip")
            
            if str(df.iloc[0, 0]).strip().lower() == "match_id":
                df.columns = df.iloc[0].astype(str).str.strip().str.lower()
                df = df[1:].reset_index(drop=True)
            else:
                base_cols = [
                    "match_id", "season", "start_date", "venue", "innings", "ball", "ball_2", 
                    "batting_team", "bowling_team", "striker", "non_striker", "bowler", 
                    "runs_off_bat", "extras", "wides", "noballs", "byes", "legbyes", "penalty", 
                    "wicket_type", "player_out"
                ]
                col_names = base_cols[:len(df.columns)]
                if len(df.columns) > len(base_cols):
                    col_names += [f"col_{i}" for i in range(len(base_cols), len(df.columns))]
                df.columns = col_names

        except Exception:
            continue

        df["season"] = df["season"].astype(str).str.split('/').str[0]
        df["season"] = pd.to_numeric(df["season"], errors="coerce")
        df["innings"] = pd.to_numeric(df["innings"], errors="coerce")
        df["runs_off_bat"] = pd.to_numeric(df["runs_off_bat"], errors="coerce").fillna(0)
        df["extras"] = pd.to_numeric(df["extras"], errors="coerce").fillna(0)

        df = df.dropna(subset=["season", "innings", "ball", "venue"])
        
        if len(df) == 0:
            continue
        
        season = int(df["season"].iloc[0])
        
        # "Wankhede Stadium, Mumbai" becomes "Wankhede Stadium"
        venue = str(df["venue"].iloc[0]).split(',')[0].strip()

        if season >= 2023:
            continue

        if "over" not in df.columns:
            df["over"] = df["ball"].astype(str).str.split(".").str[0].astype(int)

        for inning in [1, 2]:
            df_inn = df[df["innings"] == inning]
            if len(df_inn) == 0:
                continue

            is_chasing = 1 if inning == 2 else 0

            df_3 = df_inn[df_inn["over"] < 3]
            if len(df_3) == 0:
                continue

            R3 = (df_3["runs_off_bat"] + df_3["extras"]).sum()
            W3 = df_3["wicket_type"].notna().sum() if "wicket_type" in df_3.columns else 0

            last_over = df_3[df_3["over"] == 2]
            last_runs = (last_over["runs_off_bat"] + last_over["extras"]).sum()
            momentum = last_runs - (R3 / 3.0)

            dot_balls = ((df_3["runs_off_bat"] == 0) & (df_3["extras"] == 0)).sum()
            boundaries = (df_3["runs_off_bat"] >= 4).sum()

            df_6 = df_inn[df_inn["over"] < 6]
            Y6 = (df_6["runs_off_bat"] + df_6["extras"]).sum()

            raw_training_data.append({
                "W3": W3, "momentum": momentum, "dot_balls": dot_balls, 
                "boundaries": boundaries, "is_chasing": is_chasing,
                "venue": venue, "Y6": Y6, "R3": R3, "season": season
            })

if not raw_training_data:
    raise ValueError("0 matches found! Adjust the 'season >= 2023' filter.")

all_venues = sorted(list(set(d["venue"] for d in raw_training_data)))

baseline_venue = all_venues[0]
unique_venues = all_venues[1:] 

print(f"Found {len(all_venues)} unique venues!")
print(f"Using '{baseline_venue}' as the baseline (0 for all other venue columns).")

X_list, Y_list, R3_list, weights = [], [], [], []

for d in raw_training_data:
    row = [1, d["W3"], d["momentum"], d["dot_balls"], d["boundaries"], d["is_chasing"]]
    
    venue_vec = [1 if v == d["venue"] else 0 for v in unique_venues]
    row.extend(venue_vec)
    
    X_list.append(row)
    Y_list.append(d["Y6"])
    R3_list.append(d["R3"])
    weights.append(d["season"])

X = np.array(X_list)
Y = np.array(Y_list)
R3_arr = np.array(R3_list)
baseline = 2 * R3_arr
Y = Y - baseline
weights = np.array(weights, dtype=float)

print("Training Samples Built:", len(X))

weights = np.exp((weights - weights.max()) / 1.5)

W_sqrt = np.sqrt(weights)
X_w = X * W_sqrt[:, None]
Y_w = Y * W_sqrt

lambda_ridge = 15.0 
penalty_matrix = np.sqrt(lambda_ridge) * np.eye(X_w.shape[1])
penalty_matrix[0, 0] = 0  

X_aug = np.vstack([X_w, penalty_matrix])
Y_aug = np.concatenate([Y_w, np.zeros(X_w.shape[1])])

beta, _, _, _ = lstsq(X_aug, Y_aug)
print("Model trained successfully! Beta array length:", len(beta))

Scanning files and extracting features...
Found 39 unique venues!
Using 'Arun Jaitley Stadium' as the baseline (0 for all other venue columns).
Training Samples Built: 1898
Model trained successfully! Beta array length: 44


Using data from a particular file here we can check our result by multiplying the required feature vector with values from the match with beta to obtain our estimated 6 over score

In [ ]:
def predict_from_file(match_file, beta, unique_venues):
    import pandas as pd
    import numpy as np
    import random

    df = pd.read_csv(match_file, header=None, engine="python", on_bad_lines="skip")
    
    if str(df.iloc[0, 0]).strip().lower() == "match_id":
        df.columns = df.iloc[0].astype(str).str.strip().str.lower()
        df = df[1:].reset_index(drop=True)
    else:
        base_cols = [
            "match_id", "season", "start_date", "venue", "innings", "ball", "ball_2", 
            "batting_team", "bowling_team", "striker", "non_striker", "bowler", 
            "runs_off_bat", "extras", "wides", "noballs", "byes", "legbyes", "penalty", 
            "wicket_type", "player_out"
        ]
        col_names = base_cols[:len(df.columns)]
        if len(df.columns) > len(base_cols):
            col_names += [f"col_{i}" for i in range(len(base_cols), len(df.columns))]
        df.columns = col_names

    if "runs_off_bat" not in df.columns and "batsman_runs" in df.columns:
        df = df.rename(columns={"batsman_runs": "runs_off_bat"})
    if "extras" not in df.columns and "extra_runs" in df.columns:
        df = df.rename(columns={"extra_runs": "extras"})

    if "runs_off_bat" not in df.columns or "venue" not in df.columns:
        return None, None 

    df["runs_off_bat"] = pd.to_numeric(df["runs_off_bat"], errors="coerce").fillna(0)
    df["extras"] = pd.to_numeric(df["extras"], errors="coerce").fillna(0)
    df["innings"] = pd.to_numeric(df["innings"], errors="coerce")

    df = df.dropna(subset=["innings", "ball", "venue"])

    if "over" not in df.columns:
        df["over"] = df["ball"].astype(str).str.split(".").str[0].astype(int)

    valid_innings = [i for i in df["innings"].unique() if i in [1, 2]]
    if not valid_innings:
        return None, None
        
    chosen_inning = random.choice(valid_innings)
    df = df[df["innings"] == chosen_inning]
    
    is_chasing = 1 if chosen_inning == 2 else 0
    
    venue = str(df["venue"].iloc[0]).split(',')[0].strip()

    df_3 = df[df["over"] < 3]

    R3 = (df_3["runs_off_bat"] + df_3["extras"]).sum()
    W3 = df_3["wicket_type"].notna().sum() if "wicket_type" in df_3.columns else 0

    last_over = df_3[df_3["over"] == 2]
    last_runs = (last_over["runs_off_bat"] + last_over["extras"]).sum()
    momentum = last_runs - (R3 / 3.0)

    dot_balls = ((df_3["runs_off_bat"] == 0) & (df_3["extras"] == 0)).sum()
    boundaries = (df_3["runs_off_bat"] >= 4).sum()

    row = [1, W3, momentum, dot_balls, boundaries, is_chasing]
    venue_vec = [1 if v == venue else 0 for v in unique_venues]
    row.extend(venue_vec)
    
    x = np.array(row)
    y_pred = float(np.dot(x , beta) + 2 * R3)

    df_6 = df[df["over"] < 6]
    y_true = (df_6["runs_off_bat"] + df_6["extras"]).sum()

    return y_pred, y_true

We use random matches from 2026 to check our rmse values in the next 2 boxes

In [ ]:
import os
import random
import pandas as pd
import numpy as np

valid_test_files = []
data_path = r"C:\Users\SM MUKUNTHAN\ipl-powerplay-predictor\data\ipl2"

if os.path.exists(data_path):
    for f in os.listdir(data_path):
        if f.endswith(".csv") and f.split(".")[0].isdigit():
            match_file = os.path.join(data_path, f)
            try:
                df_temp = pd.read_csv(match_file, header=None, nrows=2, engine="python", on_bad_lines="skip")
                
                if str(df_temp.iloc[0, 0]).strip().lower() == "match_id":
                    season_raw = str(df_temp.iloc[1, 1]).split('/')[0]
                else:
                    season_raw = str(df_temp.iloc[0, 1]).split('/')[0]
                
                season = int(pd.to_numeric(season_raw, errors="coerce"))
                
                if season >= 2023: 
                    valid_test_files.append(match_file)
            except:
                continue

def test_random_match(valid_files, beta, unique_venues):
    if not valid_files:
        print("No test matches found! Check the year filter.")
        return 0
    
    for _ in range(5):
        match_file = random.choice(valid_files)
        y_pred, y_true = predict_from_file(match_file, beta, unique_venues)
        
        if y_pred is not None:
            print(f"Match file: {os.path.basename(match_file)}")
            print(f"Predicted 6-over score: {round(y_pred, 2)}")
            print(f"Actual 6-over score: {y_true}")
            print(f"Error: {round(abs(y_pred - y_true), 2)}\n")
            return abs(y_pred - y_true)
            
    print("Encountered multiple malformed files in a row. Skipping.")
    return 0

In [ ]:
import random
import numpy as np

random.seed(42) 

tot = 0
iterations = 50

if valid_test_files:
    for i in range(iterations):
        e = test_random_match(valid_test_files, beta, unique_venues)
        tot += e**2
        
    rmse = np.sqrt(tot / iterations)
    print("================================")
    print(f"Final RMSE over {iterations} matches: {round(rmse, 2)}")
else:
    print("Could not calculate RMSE. No test files available.")

Match file: 1359532.csv
Predicted 6-over score: 54.07
Actual 6-over score: 56
Error: 1.93

Match file: 1426307.csv
Predicted 6-over score: 48.83
Actual 6-over score: 61
Error: 12.17

Match file: 1426279.csv
Predicted 6-over score: 69.45
Actual 6-over score: 61
Error: 8.45

Match file: 1359527.csv
Predicted 6-over score: 51.29
Actual 6-over score: 58
Error: 6.71

Match file: 1473510.csv
Predicted 6-over score: 60.29
Actual 6-over score: 65
Error: 4.71

Match file: 1359490.csv
Predicted 6-over score: 52.37
Actual 6-over score: 51
Error: 1.37

Match file: 1426276.csv
Predicted 6-over score: 45.47
Actual 6-over score: 45
Error: 0.47

Match file: 1529283.csv
Predicted 6-over score: 66.79
Actual 6-over score: 65
Error: 1.79

Match file: 1529312.csv
Predicted 6-over score: 51.89
Actual 6-over score: 54
Error: 2.11

Match file: 1529304.csv
Predicted 6-over score: 45.24
Actual 6-over score: 49
Error: 3.76

Match file: 1426277.csv
Predicted 6-over score: 46.49
Actual 6-over score: 45
Error: 1.49

As an improvement, we can try using additional features or adding priors using team info or even batsmen and bowler info